In [ ]:
import requests
import json
import time
import os

# --- CẤU HÌNH ---
API_KEY = 'ba39c73252cd9fb0849949da47454e7d'
BASE_URL = 'https://api.themoviedb.org/3'
OUTPUT_FILE = 'tmdb_movies_final.json'
PROCESSED_IDS_FILE = 'processed_ids.log'

# Giới hạn cứng
MAX_PAGES_WANTED = 500
def get_existing_ids(filename):
    if not os.path.exists(filename):
        return set()
    with open(filename, 'r') as f:
        return {int(line.strip()) for line in f}

def get_movie_details(movie_id):

    details_url = f"{BASE_URL}/movie/{movie_id}"
    params_details = {'api_key': API_KEY, 'language': 'vi-VN'}
    try:
        details_res = requests.get(details_url, params=params_details)
        details_res.raise_for_status()
        details = details_res.json()
        time.sleep(0.1)
    except requests.exceptions.RequestException as e:
        print(f"  -> Lỗi khi lấy chi tiết phim ID {movie_id}: {e}")
        return None

    credits_url = f"{BASE_URL}/movie/{movie_id}/credits"
    params_credits = {'api_key': API_KEY, 'language': 'vi-VN'}
    try:
        credits_res = requests.get(credits_url, params=params_credits)
        credits_res.raise_for_status()
        credits = credits_res.json()
    except requests.exceptions.RequestException as e:
        print(f"  -> Lỗi khi lấy credits phim ID {movie_id}: {e}")
        return None

    director = next((member['name'] for member in credits['crew'] if member['job'] == 'Director'), None)
    main_actors = [actor['name'] for actor in credits.get('cast', [])[:3]]
    genres = [genre['name'] for genre in details.get('genres', [])[:3]]

    collection_info = None
    if details.get('belongs_to_collection'):
        collection_info = {
            'id': details['belongs_to_collection']['id'],
            'name': details['belongs_to_collection']['name']
        }

    return {
        'tmdb_id': movie_id, 'ten_phim_tv': details.get('title'),
        'ten_phim_ta': details.get('original_title'), 'dao_dien': director,
        'nhan_vat_chinh': main_actors, 'the_loai': genres,
        'nam_phat_hanh': details.get('release_date', ' ')[0:4],
        'mo_ta': details.get('overview'), 'diem_danh_gia': details.get('vote_average'),
        'series_info': collection_info
    }


# --- HÀM CHÍNH ĐỂ CHẠY CRAWLER ---
if __name__ == "__main__":
    processed_ids = get_existing_ids(PROCESSED_IDS_FILE)
    print(f"Đã tìm thấy {len(processed_ids)} phim đã được crawl trước đó. Bắt đầu tiếp tục...")

    all_movies_data = []
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            all_movies_data = json.load(f)


    page_num = 1
    # Khởi tạo total_pages, sẽ được cập nhật sau lần gọi API đầu tiên
    total_pages_from_api = page_num

    # Vòng lặp sẽ chạy khi trang hiện tại vẫn còn hợp lệ
    while page_num <= total_pages_from_api:
        print(f"\n--- Đang lấy dữ liệu từ trang {page_num}/{total_pages_from_api} ---")

        discover_url = f"{BASE_URL}/discover/movie"
        params = {
            'api_key': API_KEY, 'language': 'vi-VN',
            'sort_by': 'popularity.desc', 'include_adult': 'false',
            'include_video': 'false', 'page': page_num
        }

        try:
            response = requests.get(discover_url, params=params)
            response.raise_for_status()
            data = response.json()
            movies_on_page = data.get('results', [])

            # CẬP NHẬT TỔNG SỐ TRANG TỪ API
            # Lấy tổng số trang thực tế mà API cung cấp
            api_pages = data.get('total_pages', 0)
            # Giới hạn cuối cùng là số nhỏ hơn giữa số trang API có và số trang ta muốn
            total_pages_from_api = min(api_pages, MAX_PAGES_WANTED)

            # ĐIỀU KIỆN DỪNG: Nếu API trả về một trang không có phim nào
            if not movies_on_page:
                print("Trang này không có kết quả. Dừng crawl.")
                break

            for movie_summary in movies_on_page:
                movie_id = movie_summary['id']
                if movie_id in processed_ids:
                    print(f"Bỏ qua phim ID {movie_id} (đã được crawl).")
                    continue

                print(f"Đang xử lý phim: '{movie_summary.get('title', 'N/A')}' (ID: {movie_id})")
                details = get_movie_details(movie_id)

                if details:
                    all_movies_data.append(details)
                    processed_ids.add(movie_id)
                    with open(PROCESSED_IDS_FILE, 'a') as f:
                        f.write(f"{movie_id}\n")

                time.sleep(0.3)

            with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
                json.dump(all_movies_data, f, ensure_ascii=False, indent=4)
            print(f"Đã lưu {len(all_movies_data)} phim vào file {OUTPUT_FILE}")

            # Chuyển sang trang tiếp theo
            page_num += 1

        except requests.exceptions.RequestException as e:
            print(f"Lỗi nghiêm trọng khi lấy dữ liệu trang {page_num}: {e}. Dừng lại.")
            break
        except Exception as e:
            print(f"Một lỗi không xác định đã xảy ra: {e}")
            break

    print(f"\n--- HOÀN TẤT ---")
    print(f"Đã crawl đến trang cuối cùng hoặc đạt giới hạn.")
    print(f"Tổng cộng đã crawl được {len(all_movies_data)} bộ phim.")
    print(f"Dữ liệu được lưu tại: {OUTPUT_FILE}")

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
Đang xử lý phim: 'The Depraved' (ID: 83937)
Đang xử lý phim: 'Lực Lượng Bóng Đêm' (ID: 757725)
Đang xử lý phim: 'LAIDBACKERS-レイドバッカーズ-' (ID: 572647)
Đã lưu 2480 phim vào file tmdb_movies_final.json

--- Đang lấy dữ liệu từ trang 125/500 ---
Đang xử lý phim: 'Extase' (ID: 43124)
Đang xử lý phim: 'Obžalovaný' (ID: 270749)
Đang xử lý phim: 'LolliLove' (ID: 19719)
Đang xử lý phim: 'Những Kẻ Khờ Mộng Mơ' (ID: 313369)
Đang xử lý phim: 'Rừng Săn Người' (ID: 697799)
Đang xử lý phim: 'Febbre da cavallo - La mandrakata' (ID: 38529)
Đang xử lý phim: 'Mars' (ID: 916588)
Đang xử lý phim: '痴漢電車 あの娘にタッチ' (ID: 981044)
Đang xử lý phim: 'Vaazhkai Alaigal' (ID: 261633)
Đang xử lý phim: 'Hành Trình Giải Cứu Tình Yêu' (ID: 228326)
Đang xử lý phim: 'Wilson' (ID: 346681)
Đang xử lý phim: 'Thiện Xạ' (ID: 7485)
Đang xử lý phim: 'Bugso' (ID: 1053630)
Đang xử lý phim: 'Mandao Returns' (ID: 778596)
Đang xử lý phim: 'Khoái Cảm' (ID: 592695)
Đang xử lý phim: 